# Run the Stage 1 GHI pipeline (orchestrator)

[!] writing to `*_v2` tables so his originals are never touched. [!] 

```
build_structured_data  →  build_split_days  →  build_ghi_model  →  build_all_uncurtailedpv
        (_v2)                   (_v2)                (_v2)                  (_v2)
```
**Cost note:** the structured_data build scans `ts` (billions of rows). Run the
**tiny test slice first** (one month, one site-part) and confirm it works before
scaling to the full year. Athena bills by data scanned.

In [ ]:
# Bootstrap: paths + imports
import sys, pathlib
SHARED = pathlib.Path(r"../../shared")
STAGE1  = pathlib.Path(r"../stage1_ghi_pipeline")
sys.path.insert(0, str(SHARED))
sys.path.insert(0, str(STAGE1))

from aws_config import aq            # your existing Athena helper
from ciccada_config import SAI       # 'solar_analytics_iceberg'

import build_structured_data   as b1
import build_split_days        as b2
import build_ghi_model         as b3
import build_mape_quality_gate as b3b
import build_all_uncurtailedpv as b4

print("Target tables (note the _v2 suffix. Original results untouched):")
print(" ", b1.TARGET)
print(" ", b2.TARGET)
print(" ", b3.TARGET)
print(" ", b4.TARGET)

## Step 1. Structured_data_v2

In [ ]:
# 1a. Create the empty table (safe: drops & recreates only the _v2 table)
print(b1.create_table(aq, database=SAI))

In [ ]:
# 1b. TEST SLICE FIRST. 
# One month, one of 8 site-parts.
# Confirm this completes and validate() looks sane BEFORE the full run.
b1.run_slice(aq, database=SAI, year=2024, months=[1], n_parts=8, parts=[0])

In [ ]:
# 1c. Validate the test slice
b1.validate(aq, database=SAI)

In [ ]:
# 1d. FULL RUN. all months, all 8 site-parts, for the year(s) dedfined in the pipeline config.
#     Only run this once the test slice looks right. This is the expensive one.
#     (Re-running create_table first would wipe the test slice — that's expected;
#      the full run below reloads everything cleanly.)
print(b1.create_table(aq, database=SAI))
# Both years, 2024 and 2025:
b1.run_slice(aq, database=SAI, year=2024, months=range(1, 13), n_parts=8)
b1.run_slice(aq, database=SAI, year=2025, months=range(1, 13), n_parts=8)
b1.validate(aq, database=SAI)

## Step 2. split_days_v2

In [ ]:
print(b2.create_table(aq, database=SAI))
print(b2.run(aq, database=SAI))
b2.validate(aq, database=SAI)

## Step 3. pv_ghi_norm_model_v2

In [ ]:
print(b3.create_table(aq, database=SAI))
b3.run_year(aq, database=SAI, year=2024)
b3.run_year(aq, database=SAI, year=2025)
b3.validate(aq, database=SAI)

In [ ]:
# Step 3b. MAPE quality gate, now saving an auditable CSV
MAPE_CSV = "mape_under50_sites.csv"
mape_df, good_sites = b3b.run(aq, database=SAI, csv_path=MAPE_CSV)

## Step 4. all_uncurtailedpv_v2

In [ ]:
MAPE_CSV = r"mape_under50_sites.csv" 
print(b4.create_table(aq, database=SAI))
b4.run_year(aq, database=SAI, year=2024, mape_csv_path=MAPE_CSV, n_parts=3)
b4.run_year(aq, database=SAI, year=2025, mape_csv_path=MAPE_CSV, n_parts=3)
b4.validate(aq, database=SAI)

## Done. Stage 1 rebuilt